# 01 — Exploratory Data Analysis

Sanity-check the dataset against the paper's reported counts:
- 27,895 traders tracked over 46 weeks
- 1,229,439 observations at the trader-week level

Also: keep_country investigation, missingness audit, column diagnostics.

In [16]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')

from src.data_loading import (
    load_raw_data, clean_data, verify_counts, print_cleaning_summary,
    save_processed, load_processed, verify_column_diagnostics,
    get_missingness_report, EXPECTED_OBS_PAPER, EXPECTED_TRADERS_PAPER,
)
from src.viz import set_journal_style, COLORS

set_journal_style()
print('Imports OK')

Imports OK


## 1. Load Raw Data

In [17]:
data_path = '../trader_data_exl.csv' if os.path.exists('../trader_data_exl.csv') else 'trader_data_exl.csv'
df_raw = load_raw_data(data_path)
print(f'Raw shape: {df_raw.shape}')
print(f'Columns: {list(df_raw.columns)}')
print(f'Dtypes:\n{df_raw.dtypes}')

2026-08-19 09:46:50,890 [INFO] src.data_loading: Loading raw data from ../trader_data_exl.csv ...
2026-08-19 09:46:58,337 [INFO] src.data_loading: Raw data loaded: 1,314,404 rows, 45 columns
D:\Social Trading Causal ML\CODE repo\11. anti\notebooks\..\src\data_loading.py:66: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col] = pd.to_datetime(df[col], errors="coerce")
2026-08-19 09:46:58,597 [INFO] src.data_loading:   Converted 'joining_date' to datetime — 1,289,794 valid dates
D:\Social Trading Causal ML\CODE repo\11. anti\notebooks\..\src\data_loading.py:66: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors="coerce")
2026-08-19 09:46:58,811 [INFO] src.data_loading:   Converted 'max_period_date'

Raw shape: (1314404, 45)
Columns: ['trader_name', 'joining_date', 'user_country_name', 'period', 'trader_id', 'guru_count', 'total_allocation_to_gurus', 'learner_count', 'total_allocation_received', 'same_country_learner_count', 'learner_country_count', 'max_period_date', 'min_period_date', 'total_follower_count', 'same_country_follower_count', 'follower_country_count', 'followed_count', 'total_trxn', 'total_gain', 'total_self_trxn', 'total_self_gain', 'total_copied_trxn', 'total_copied_gain', 'greater_than_ten', 'log_follower_count', 'log_followed_count', 'log_guru_count', 'log_learner_count', 'std_gain_diff', 'gain_difference', 'total_audience', 'log_total_audience', 'joining_date2', 'foreign_country_follower_count', 'log_foreign_followers', 'log_domestic_followers', 'foreign_country_learner_count', 'log_foreign_learners', 'log_domestic_learners', 'active_followed_trader_count', 'log_active_followed_trader_count', 'keep_country', 'unique_followed_count', 'trader', 'unique_learner_cou

## 2. Missingness Report

In [18]:
miss = get_missingness_report(df_raw)
print('=== COLUMNS WITH ANY MISSING VALUES ===')
print(miss[miss['n_missing'] > 0].to_string(index=False))
print()
print(f"learner_country_count: {miss.loc[miss['column']=='learner_country_count', 'pct_missing'].values[0]:.1%} missing → EXCLUDED")

=== COLUMNS WITH ANY MISSING VALUES ===
               column  n_missing  pct_missing          dtype
learner_country_count    1264635     0.962136        float64
    user_country_name      24610     0.018723         object
         joining_date      24610     0.018723 datetime64[ns]
        joining_date2      24610     0.018723 datetime64[ns]

learner_country_count: 96.2% missing → EXCLUDED


## 3. Verify: Are joining_date/joining_date2/user_country_name nulls the same rows?

In [19]:
jd_null = df_raw['joining_date'].isna()
jd2_null = df_raw['joining_date2'].isna() if 'joining_date2' in df_raw.columns else jd_null
ucn_null = df_raw['user_country_name'].isna()

print(f'joining_date null:       {jd_null.sum():>10,}')
print(f'joining_date2 null:      {jd2_null.sum():>10,}')
print(f'user_country_name null:  {ucn_null.sum():>10,}')
print(f'All three null (same):   {(jd_null & jd2_null & ucn_null).sum():>10,}')
print(f'Any of three null:       {(jd_null | jd2_null | ucn_null).sum():>10,}')

joining_date null:           24,610
joining_date2 null:          24,610
user_country_name null:      24,610
All three null (same):       24,610
Any of three null:           24,610


## 4. Check for Fully-Blank Trailer Rows

In [20]:
fully_blank = df_raw.isnull().all(axis=1)
print(f'Fully-blank rows: {fully_blank.sum():,}')
if fully_blank.sum() > 0:
    print('Sample of fully-blank rows:')
    display(df_raw[fully_blank].head())
else:
    print('No fully-blank trailer rows found in this file.')

Fully-blank rows: 0
No fully-blank trailer rows found in this file.


## 5. Excel Truncation Check

In [21]:
n_rows = len(df_raw) - fully_blank.sum()
print(f'Data rows (non-blank): {n_rows:,}')
print(f'Paper reports:         {EXPECTED_OBS_PAPER:,}')
print(f'Excel row limit:       1,048,575')
if n_rows == 1_048_575:
    print('\n⚠ FILE IS TRUNCATED AT EXCEL ROW LIMIT')
    print(f'  Missing rows: {EXPECTED_OBS_PAPER - n_rows:,}')
print(f'\nUnique trader_ids: {df_raw["trader_id"].nunique():,} (paper: {EXPECTED_TRADERS_PAPER:,})')
print(f'Period range: {df_raw["period"].min()} — {df_raw["period"].max()}')

Data rows (non-blank): 1,314,404
Paper reports:         1,229,439
Excel row limit:       1,048,575

Unique trader_ids: 28,574 (paper: 27,895)
Period range: 30 — 75


## 6. keep_country Investigation

In [22]:
# Check 1: Fraction and distribution
kc = df_raw['keep_country']
print(f'keep_country value counts:')
print(kc.value_counts())
print(f'\nFraction kc=0: {(kc==0).mean():.2%}')

# Check 2: Is user_country_name null for kc=0?
kc0 = df_raw[kc == 0]
kc1 = df_raw[kc == 1]
print(f'\nkc=0 rows with null user_country_name: {kc0["user_country_name"].isna().mean():.1%}')
print(f'kc=1 rows with null user_country_name: {kc1["user_country_name"].isna().mean():.1%}')

# Show sample kc=0 rows
print('\nSample kc=0 rows:')
display(kc0[['trader_id', 'period', 'user_country_name', 'total_gain', 'total_self_trxn', 'keep_country']].head(10))

#  1,229,439 - paper
# 1,287,824 - 
# 1,196,100

keep_country value counts:
keep_country
1    1222680
0      91724
Name: count, dtype: int64

Fraction kc=0: 6.98%

kc=0 rows with null user_country_name: 26.8%
kc=1 rows with null user_country_name: 0.0%

Sample kc=0 rows:


,trader_id,period,user_country_name,total_gain,total_self_trxn,keep_country
0,1,30,NaN,0.0,0,0
1,1,31,NaN,0.0,0,0
2,1,32,NaN,0.0,0,0
3,1,33,NaN,0.0,0,0
4,1,34,NaN,0.0,0,0
5,1,35,NaN,0.0,0,0
6,1,36,NaN,0.0,0,0
7,1,37,NaN,0.0,0,0
8,1,38,NaN,0.0,0,0
9,1,39,NaN,0.0,0,0


In [23]:
1229439 - 1196100

33339

In [24]:
# Check 3: Compare outcomes
print('=== Comparison: kc=0 vs kc=1 ===')
for col in ['total_gain', 'total_self_trxn', 'total_trxn']:
    print(f'\n{col}:')
    print(f'  kc=0: mean={kc0[col].mean():.2f}, median={kc0[col].median():.2f}')
    print(f'  kc=1: mean={kc1[col].mean():.2f}, median={kc1[col].median():.2f}')

=== Comparison: kc=0 vs kc=1 ===

total_gain:
  kc=0: mean=0.75, median=0.00
  kc=1: mean=-1.27, median=0.00

total_self_trxn:
  kc=0: mean=0.13, median=0.00
  kc=1: mean=0.15, median=0.00

total_trxn:
  kc=0: mean=0.57, median=0.00
  kc=1: mean=0.69, median=0.00


## 7. Column Diagnostics

In [25]:
diag = verify_column_diagnostics(df_raw)
print('Column diagnostics:')
for k, v in diag.items():
    print(f'  {k}: {v}')

2026-08-19 09:47:01,526 [INFO] src.data_loading: std_gain_diff ↔ gain_difference correlation: 1.0000


Column diagnostics:
  std_gain_diff_vs_gain_difference_corr: 1.0
  total_audience_equals_follower_plus_learner: False
  total_audience_match_rate: 0.8438691604712097
  total_audience_diff_stats: {'count': 1314404.0, 'mean': -6.4684427314585164, 'std': 122.53743615983478, 'min': -9679.0, '25%': 0.0, '50%': 0.0, '75%': 0.0, 'max': 0.0}


## 8. Clean Data and Verify

In [26]:
df, stats = clean_data(df_raw, apply_keep_country=None)
print_cleaning_summary(stats)
checks = verify_counts(stats)
print(f'\nVerification: {checks}')

2026-08-19 09:47:01,929 [INFO] src.data_loading: No fully-blank trailer rows found in this file
2026-08-19 09:47:02,147 [INFO] src.data_loading: keep_country auto-decision: filter to kc=1 (fraction kc=0: 7.0%)
2026-08-19 09:47:02,636 [INFO] src.data_loading: Applied keep_country=1 filter: removed 91,724 rows (7.0%), 1,994 traders
2026-08-19 09:47:02,685 [INFO] src.data_loading: joining_date2 matches joining_date: 100.0000%
2026-08-19 09:47:02,902 [INFO] src.data_loading: Dropped 'joining_date2' (confirmed duplicate)
2026-08-19 09:47:02,907 [INFO] src.data_loading: Dropping 'learner_country_count' — 96.3% missing (excluded from all downstream analysis)
2026-08-19 09:47:03,218 [INFO] src.data_loading:   ✓ row_count_matches_paper
2026-08-19 09:47:03,218 [INFO] src.data_loading:   ✓ period_range_correct
2026-08-19 09:47:03,218 [INFO] src.data_loading:   ✓ median_rows_near_46



DATA CLEANING SUMMARY
  Total raw rows:                 1,314,404
  Fully-blank rows removed:               0
  Rows after blank removal:       1,314,404
  keep_country rows removed:         91,724
  Final row count:                1,222,680
  Unique trader_ids:                 26,580
  Period range:                30 — 75 (46 unique periods)

  Rows-per-trader distribution:
    Mean:   46.0
    Median: 46
    Min:    46
    Max:    46
    Std:    0.0
    P10: 46      P25: 46      P50: 46      P75: 46      P90: 46  

  joining_date2 match rate:    100.0000%
  learner_country_count:       96.3% missing → DROPPED


Verification: {'row_count_matches_paper': True, 'period_range_correct': True, 'median_rows_near_46': True}


## 9. Panel Structure Verification

In [27]:
rows_per_trader = df.groupby('trader_id').size()
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(rows_per_trader, bins=46, color=COLORS['primary'], alpha=0.7, edgecolor='white')
ax.axvline(rows_per_trader.median(), color=COLORS['secondary'], linestyle='--', label=f'Median={rows_per_trader.median():.0f}')
ax.set_xlabel('Periods per Trader')
ax.set_ylabel('Number of Traders')
ax.set_title('Panel Balance: Periods per Trader')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Rows-per-trader: mean={rows_per_trader.mean():.1f}, median={rows_per_trader.median():.0f}')

Rows-per-trader: mean=46.0, median=46


C:\Users\shara\AppData\Local\Temp\ipykernel_20784\3567100447.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [28]:
# Save for downstream notebooks
save_processed(df, '../data/processed/trader_panel.parquet')
print('Saved cleaned data to parquet.')

2026-08-19 09:47:06,214 [INFO] src.data_loading: Saved processed data to ../data/processed/trader_panel.parquet (8.3 MB)


Saved cleaned data to parquet.
